In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import distance_transform_edt
from tqdm import tqdm

# --- CONFIGURATION ---
CSV_PATH = r'cleaned_dataset\new_csv\train_full.csv'
ORIGINAL_MASKS_DIR = r'cleaned_dataset\cut\train_masks'
OUTPUT_BASE = 'prepare'
FINAL_SIZE = (512, 512)

# Create output directories
folders = ['full_masks', 'sdf', 'weight_maps', 'partial_masks_resized']
for folder in folders:
    os.makedirs(os.path.join(OUTPUT_BASE, folder), exist_ok=True)

def parse_pt(pt_str):
    try:
        return list(map(int, pt_str.strip('"').split(',')))
    except:
        return None

def parse_curve(curve_str):
    try:
        return [list(map(int, p.split(','))) for p in curve_str.split('|')]
    except:
        return []

def compute_sdf(mask, clip_dist=20):
    mask = mask.astype(bool)
    pos_dist = distance_transform_edt(mask)
    neg_dist = distance_transform_edt(~mask)
    sdf = pos_dist - neg_dist
    return np.clip(sdf, -clip_dist, clip_dist) / clip_dist

# Load Dataset
df = pd.read_csv(CSV_PATH)

print(f"Preparing {len(df)} images (no resizing, already aligned)...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    img_name = row['image_path']
    
    partial_mask = cv2.imread(
        os.path.join(ORIGINAL_MASKS_DIR, img_name),
        cv2.IMREAD_GRAYSCALE
    )
    
    if partial_mask is None:
        continue

    # --- Ensure binary ---
    partial_mask = (partial_mask > 127).astype(np.uint8) * 255

    # --- Parse reconstruction data ---
    c_l = parse_pt(row['cutoff_l'])
    c_r = parse_pt(row['cutoff_r'])
    curve_pts = parse_curve(row['curve'])

    # --- Reconstruction layer ---
    recon_layer = np.zeros_like(partial_mask, dtype=np.uint8)

    if c_l and c_r and curve_pts:
        poly = np.array([c_l] + curve_pts + [c_r], dtype=np.int32)

        # Fill reconstructed region
        cv2.fillPoly(recon_layer, [poly], 255)

        # Seal boundary
        cv2.polylines(recon_layer, [poly], isClosed=True, color=255, thickness=2)

    # --- Merge ---
    full_mask = cv2.bitwise_or(partial_mask, recon_layer)

    # --- Training targets ---
    gap = cv2.subtract(full_mask, partial_mask)

    weight_map = np.ones(FINAL_SIZE, dtype=np.float32)
    weight_map[gap > 0] = 5.0

    sdf_target = compute_sdf(full_mask)

    # --- Save ---
    base = os.path.splitext(img_name)[0]

    cv2.imwrite(os.path.join(OUTPUT_BASE, 'full_masks', f"{base}.png"), full_mask)
    cv2.imwrite(os.path.join(OUTPUT_BASE, 'partial_masks_resized', f"{base}.png"), partial_mask)

    np.save(os.path.join(OUTPUT_BASE, 'weight_maps', f"{base}_weight.npy"), weight_map)
    np.save(os.path.join(OUTPUT_BASE, 'sdf', f"{base}_sdf.npy"), sdf_target)

print("\nPreparation Complete! (No padding/resizing)")

Preparing 204 images (no resizing, already aligned)...


100%|██████████| 204/204 [00:08<00:00, 24.53it/s]


Preparation Complete! (No padding/resizing)
